In [ ]:
%python
# ── PASS 1 : Expire changed records ──────────────────────────────
spark.sql("""
    MERGE INTO catalog_name.silver_db.employee_scd2 AS trg
    USING (
        SELECT src.*
        FROM catalog_name.silver_db.employee_stage AS src
        INNER JOIN catalog_name.silver_db.employee_scd2 AS cur
            ON  cur.emp_name  = src.emp_name
            AND cur.is_active = 1
        WHERE
            cur.salary      IS DISTINCT FROM src.salary      OR
            cur.designation IS DISTINCT FROM src.designation OR
            cur.department  IS DISTINCT FROM src.department
    ) AS src
    ON  trg.emp_name  = src.emp_name
    AND trg.is_active = 1

    WHEN MATCHED THEN
        UPDATE SET
            trg.is_active = 0,
            trg.end_date  = CURRENT_TIMESTAMP
""")

# ── PASS 2 : Insert new/fresh versions ───────────────────────────
spark.sql("""
    MERGE INTO catalog_name.silver_db.employee_scd2 AS trg
    USING (
        SELECT src.*
        FROM catalog_name.silver_db.employee_stage AS src
        LEFT JOIN catalog_name.silver_db.employee_scd2 AS cur
            ON  cur.emp_name  = src.emp_name
            AND cur.is_active = 1
        WHERE cur.emp_name IS NULL
    ) AS src
    ON  trg.emp_name  = src.emp_name
    AND trg.is_active = 1

    WHEN NOT MATCHED BY TARGET THEN
        INSERT (
            emp_name, hr_name, manager_name, salary,
            location, designation, department,
            doj, dob, rank,
            is_active, start_date, end_date
        )
        VALUES (
            src.emp_name, src.hr_name, src.manager_name, src.salary,
            src.location, src.designation, src.department,
            src.doj, src.dob, src.rank,
            1, CURRENT_TIMESTAMP, NULL
        )
""")